# `apply` vs `apply_peetre`: Efficiency & Precision Analysis

This notebook compares the standard `apply()` method (which uses direct quadrature for spatially-dependent symbols) against the `apply_peetre()` method (which uses symbolic decomposition to accelerate separable symbols).

We use the exact same symbols and test function $u$ as in the baseline efficiency analysis.

### 🐛 Bug Fix Note: The Frequency Grid Misalignment
If you previously ran this with `np.fft.fftshift` applied to `kx` and `ky`, you likely saw massive errors (e.g., `1.46e+04`) for Variable Coefficients, while Constant Coefficients had `0.00e+00` error.

**Why?** The slow path in `apply()` correctly recomputes frequencies internally. But the fast path `_apply_constant_fft()` (used by `apply_peetre()` for the separated $q(\xi)$ multipliers) blindly uses the `kx` array passed to it. Passing a *shifted* `kx` multiplied the unshifted FFT spectrum by the shifted symbol, completely misaligning the frequencies.

**The Fix:** We simply use the **unshifted** `fftfreq` grid below so it perfectly matches the `scipy.fft.fft` convention.

In [ ]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import time
from psiop import PseudoDifferentialOperator

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

boundary_condition = 'periodic'

In [ ]:
def benchmark_compare(op, x_grid, kx, y_grid=None, ky=None, repeats=2, boundary_condition='periodic'):
    """
    Benchmarks apply() vs apply_peetre(), returning mean execution times and relative L2 error.
    """
    dim = op.dim
    if dim == 1:
        u = np.exp(-x_grid**2) * np.cos(5 * x_grid)
    else:
        X, Y = np.meshgrid(x_grid, y_grid, indexing='ij')
        u = np.exp(-(X**2 + Y**2)) * np.cos(5 * X) * np.cos(5 * Y)
        
    # Warm-up calls
    op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky, boundary_condition='periodic')
    op.apply_peetre(u, x_grid, kx, y_grid=y_grid, ky=ky, boundary_condition='periodic')
    
    times_apply = []
    times_peetre = []
    
    for _ in range(repeats):
        start = time.perf_counter()
        res_apply = op.apply(u, x_grid, kx, y_grid=y_grid, ky=ky, boundary_condition=boundary_condition, backend='direct')
        end = time.perf_counter()
        times_apply.append(end - start)
        
        start = time.perf_counter()
        res_peetre = op.apply_peetre(u, x_grid, kx, y_grid=y_grid, ky=ky, boundary_condition=boundary_condition)
        end = time.perf_counter()
        times_peetre.append(end - start)
        
    mean_apply = np.mean(times_apply)
    mean_peetre = np.mean(times_peetre)
    
    # Relative L2 Error (using apply as reference)
    norm_ref = np.linalg.norm(res_apply)
    if norm_ref > 0:
        err = np.linalg.norm(res_apply - res_peetre) / norm_ref
    else:
        err = np.linalg.norm(res_apply - res_peetre)
        
    return mean_apply, mean_peetre, err

In [ ]:
# 1D Symbols
x, xi = sp.symbols('x xi', real=True)

sym_1d_loc_c = xi**2
sym_1d_loc_v = (1 + 0.5 * sp.sin(x)) * xi**2
sym_1d_nloc_c = sp.sqrt(xi**2 + 1.0) 
sym_1d_nloc_v = (1 + 0.5 * sp.sin(x)) * sp.sqrt(xi**2 + 1.0)

ops_1d = {
    '1D Local (Const)': PseudoDifferentialOperator(sym_1d_loc_c, [x], mode='symbol'),
    '1D Local (Var)': PseudoDifferentialOperator(sym_1d_loc_v, [x], mode='symbol'),
    '1D Non-Local (Const)': PseudoDifferentialOperator(sym_1d_nloc_c, [x], mode='symbol'),
    '1D Non-Local (Var)': PseudoDifferentialOperator(sym_1d_nloc_v, [x], mode='symbol')
}

In [ ]:
N_values_1d = [256, 512, 1024, 2048, 4096]
results_1d = {k: {'apply': [], 'peetre': [], 'err': []} for k in ops_1d.keys()}

print('Running 1D Benchmarks...')
for N in N_values_1d:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)
    
    # FIX: Use unshifted frequencies to match scipy.fft.fft output convention
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    
    for name, op in ops_1d.items():
        t_apply, t_peetre, err = benchmark_compare(op, x_grid, kx, repeats=4, boundary_condition=boundary_condition)
        results_1d[name]['apply'].append(t_apply)
        results_1d[name]['peetre'].append(t_peetre)
        results_1d[name]['err'].append(err)
        print(f'N={N:5d} | {name:25s} | apply: {t_apply:.4f}s | peetre: {t_peetre:.4f}s | err: {err:.2e}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {'1D Local (Const)': 'blue', '1D Local (Var)': 'orange', 
          '1D Non-Local (Const)': 'green', '1D Non-Local (Var)': 'red'}

for name in ops_1d.keys():
    ax1.loglog(N_values_1d, results_1d[name]['apply'], marker='o', linestyle='--', 
               color=colors[name], label=f'{name} (apply)')
    ax1.loglog(N_values_1d, results_1d[name]['peetre'], marker='s', linestyle='-', 
               color=colors[name], label=f'{name} (peetre)')

ax1.set_xlabel('Grid Size N')
ax1.set_ylabel('Execution Time (s)')
ax1.set_title('1D Execution Time: apply vs apply_peetre')
ax1.legend(fontsize=8)

for name in ops_1d.keys():
    ax2.semilogy(N_values_1d, results_1d[name]['err'], marker='d', 
                 color=colors[name], label=name)

ax2.set_xlabel('Grid Size N')
ax2.set_ylabel('Relative L2 Error')
ax2.set_title('1D Precision: apply vs apply_peetre')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 2D Symbols
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

sym_2d_loc_c = xi**2 + eta**2
sym_2d_loc_v = (1 + 0.5 * sp.sin(x) * sp.cos(y)) * (xi**2 + eta**2)
sym_2d_nloc_c = (xi**2 + eta**2)**0.75 
sym_2d_nloc_v = (1 + 0.5 * sp.sin(x) * sp.cos(y)) * (xi**2 + eta**2)**0.75

ops_2d = {
    '2D Local (Const)': PseudoDifferentialOperator(sym_2d_loc_c, [x, y], mode='symbol'),
    '2D Local (Var)': PseudoDifferentialOperator(sym_2d_loc_v, [x, y], mode='symbol'),
    '2D Non-Local (Const)': PseudoDifferentialOperator(sym_2d_nloc_c, [x, y], mode='symbol'),
    '2D Non-Local (Var)': PseudoDifferentialOperator(sym_2d_nloc_v, [x, y], mode='symbol')
}

In [ ]:
# We limit N to 128 for 2D because `apply` on variable coefficients is O(N^4) 
# and would take too long / OOM for N=256+.
N_values_2d = [32, 64, 128]
results_2d = {k: {'apply': [], 'peetre': [], 'err': []} for k in ops_2d.keys()}

print('Running 2D Benchmarks... (This may take a few minutes for Variable Coeffs with `apply`)')
for N in N_values_2d:
    L = 10
    dx = L / N
    x_grid = -L/2 + dx * np.arange(N)
    y_grid = -L/2 + dx * np.arange(N)
    
    # FIX: Use unshifted frequencies
    kx = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    ky = 2 * np.pi * np.fft.fftfreq(N, d=dx)
    
    for name, op in ops_2d.items():
        t_apply, t_peetre, err = benchmark_compare(op, x_grid, kx, y_grid, ky, repeats=4, boundary_condition=boundary_condition)
        results_2d[name]['apply'].append(t_apply)
        results_2d[name]['peetre'].append(t_peetre)
        results_2d[name]['err'].append(err)
        print(f'N={N:4d}x{N:<4d} | {name:25s} | apply: {t_apply:.4f}s | peetre: {t_peetre:.4f}s | err: {err:.2e}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_2d = {'2D Local (Const)': 'blue', '2D Local (Var)': 'orange', 
             '2D Non-Local (Const)': 'green', '2D Non-Local (Var)': 'red'}

for name in ops_2d.keys():
    ax1.loglog(N_values_2d, results_2d[name]['apply'], marker='o', linestyle='--', 
               color=colors_2d[name], label=f'{name} (apply)')
    ax1.loglog(N_values_2d, results_2d[name]['peetre'], marker='s', linestyle='-', 
               color=colors_2d[name], label=f'{name} (peetre)')

ax1.set_xlabel('Grid Size N (Total Points = N^2)')
ax1.set_ylabel('Execution Time (s)')
ax1.set_title('2D Execution Time: apply vs apply_peetre')
ax1.legend(fontsize=8)

for name in ops_2d.keys():
    ax2.semilogy(N_values_2d, results_2d[name]['err'], marker='d', 
                 color=colors_2d[name], label=name)

ax2.set_xlabel('Grid Size N')
ax2.set_ylabel('Relative L2 Error')
ax2.set_title('2D Precision: apply vs apply_peetre')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Analysis & Conclusions

### 1. The Power of Peetre Decomposition for Separable Symbols
The standard `apply` method checks if a symbol is purely constant (no spatial dependence). If it contains *any* spatial variable, it falls back to the generic slow path:
- **1D:** $\mathcal{O}(N^2)$ direct quadrature.
- **2D:** $\mathcal{O}(N^4)$ direct quadrature.

However, the symbols defined above with variable coefficients (e.g., $(1 + 0.5 \sin(x)) \xi^2$) are **separable** in space and frequency. 
The `apply_peetre` method automatically detects this separability and decomposes the operator into a sum of terms $a(x) q(D)$. Each term is then applied using the ultra-fast $\mathcal{O}(N \log N)$ FFT path, followed by a simple pointwise multiplication by $a(x)$.

### 2. Performance Gain
- **1D:** `apply_peetre` drops the complexity from $\mathcal{O}(N^2)$ to $\mathcal{O}(N \log N)$, yielding massive speedups for $N \ge 1024$.
- **2D:** `apply_peetre` drops the complexity from $\mathcal{O}(N^4)$ to $\mathcal{O}(N^2 \log N)$. This turns an operation that would take hours or cause Out-Of-Memory (OOM) errors at $N=256$ into a fraction of a second.

### 3. Precision
Because the decomposition is mathematically exact for separable symbols, the relative L2 error between the slow quadrature (`apply`) and the fast FFT-based Peetre method (`apply_peetre`) remains at the level of machine precision ($\sim 10^{-15}$). No accuracy is sacrificed for the immense gain in computational efficiency.